# Example 06: Tool Use / Function Calling

Define Python functions and pass them to the model as tools.
When the model decides a question requires computation or a lookup, it emits a tool call.
LiteRT-LM executes the function automatically and feeds the result back to the model.

This notebook defines three tools: addition, multiplication, and a mock weather lookup.

In [ ]:
!pip install litert-lm-api-nightly litert-lm -q

In [ ]:
import subprocess
subprocess.run([
    "curl", "-L",
    "https://huggingface.co/litert-community/gemma-4-E2B-it-litert-lm/resolve/main/gemma-4-E2B-it.litertlm?download=true",
    "-o", "/content/gemma-4-E2B-it.litertlm"
], check=True)

In [ ]:
import litert_lm

MODEL_PATH = "/content/gemma-4-E2B-it.litertlm"

litert_lm.set_min_log_severity(litert_lm.LogSeverity.ERROR)


def add_numbers(a: float, b: float) -> float:
    """Adds two numbers.

    Args:
        a: The first number.
        b: The second number.
    """
    return a + b


def multiply_numbers(a: float, b: float) -> float:
    """Multiplies two numbers.

    Args:
        a: The first number.
        b: The second number.
    """
    return a * b


def get_current_weather(city: str) -> str:
    """Returns a mock weather report for a given city.

    Args:
        city: The name of the city.
    """
    return f"The weather in {city} is sunny and 22\u00b0C."


tools = [add_numbers, multiply_numbers, get_current_weather]

with litert_lm.Engine(MODEL_PATH) as engine:
    with engine.create_conversation(tools=tools) as conversation:
        queries = [
            "What is 123 + 456?",
            "What is 7 multiplied by 8?",
            "What is the weather in Istanbul?",
        ]
        for query in queries:
            print(f"Q: {query}")
            response = conversation.send_message(query)
            print(f"A: {response['content'][0]['text']}\n")